In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
import pandas as pd
import time
import re

HEDEF_TOPLAM_VERI = 10000 
ARAMA_TERIMLERI = [
    "Software Engineer", "Data Scientist", "Data Analyst", 
    "Product Manager", "Human Resources", "Marketing", 
    "Finance", "Sales", "DevOps", "Cyber Security"
]
KONUM = "Turkey"

teknik_beceriler = ['python', 'sql', 'java', 'aws', 'excel', 'tableau', 'power bi', 'javascript', 'docker', 'sap', 'c#', 'react']
sosyal_beceriler = ['communication', 'leadership', 'teamwork', 'management', 'agile', 'english', 'problem solving']

def beceri_bul(metin):
    metin = str(metin).lower()
    bulunanlar = [b for b in teknik_beceriler + sosyal_beceriler if re.search(rf'\b{b}\b', metin)]
    return list(set(bulunanlar))

def max_linkedin_scrapper():
    options = Options()
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    
    print("Tarayıcı açıldı. Lütfen giriş yapın (60 sn bekleme süresi)...")
    driver.get("https://www.linkedin.com/login")
    time.sleep(60)
    
    all_jobs = []
    checkpoint_file = "linkedin_final_data.csv"

    for terim in ARAMA_TERIMLERI:
        if len(all_jobs) >= HEDEF_TOPLAM_VERI: break
        
        current_start = 0
        print(f"\n>>> YENİ POZİSYON: {terim} başlatılıyor...")

        while True: 
            search_url = f"https://www.linkedin.com/jobs/search/?keywords={terim.replace(' ', '%20')}&location={KONUM}&start={current_start}"
            driver.get(search_url)
            time.sleep(5)

            print(f"Sayfa (start={current_start}) yükleniyor ve kaydırılıyor...")
            last_height = 0
            while True:
                try:
                    job_list_panel = driver.find_element(By.CLASS_NAME, "jobs-search-results-list")
                    driver.execute_script("arguments[0].scrollTop += 800;", job_list_panel)
                    time.sleep(1)
                    new_height = driver.execute_script("return arguments[0].scrollTop;", job_list_panel)
                    if new_height == last_height: break 
                    last_height = new_height
                except:
                    break

            job_cards = driver.find_elements(By.CLASS_NAME, "job-card-container")
            print(f"Bu sayfada {len(job_cards)} ilan bulundu. Veriler çekiliyor...")

            if not job_cards:
                print(f"{terim} için çekilecek ilan kalmadı.")
                break 

            for card in job_cards:
                try:
                    driver.execute_script("arguments[0].scrollIntoView();", card)
                    card.click()
                    time.sleep(2) 

                    title = driver.find_element(By.CLASS_NAME, "job-details-jobs-unified-top-card__job-title").text.strip()
                    company = driver.find_element(By.CLASS_NAME, "job-details-jobs-unified-top-card__company-name").text.strip()
                    desc = driver.find_element(By.ID, "job-details").text
                    
                    beceriler = beceri_bul(desc)
                    
                    all_jobs.append({
                        "Unvan": title,
                        "Sirket": company,
                        "Kriterler": ", ".join(beceriler),
                        "Beceri_Sayisi": len(beceriler),
                        "Pozisyon": terim
                    })

                    if len(all_jobs) % 10 == 0:
                        pd.DataFrame(all_jobs).to_csv(checkpoint_file, index=False, encoding='utf-8-sig')
                        print(f"Toplam Veri: {len(all_jobs)}")

                except:
                    continue 

            current_start += 25
            if current_start >= 1000: 
                break

    driver.quit()
    print(f"İşlem bitti. Toplam {len(all_jobs)} veri '{checkpoint_file}' dosyasına kaydedildi.")

max_linkedin_scrapper()